# 44 · Who runs the loop: SDK helpers and vendor-run tools
Level L4 · Part 5 · professional · ~30 min · builds on 35

## Goal
Run Scout's Mars question twice on Gemini, once with your own loop and once with the SDK's automatic
function calling, list what the helper hides, run a vendor-run tool (Gemini code execution), read a
dated Anthropic server-tool request, and pick who runs the loop for three cases with a small rule.

## The idea
In lesson 35 **you** ran the loop: call the model, run the tools, send results back, stop. Two other
options exist. An **SDK helper** (Gemini's automatic function calling, Anthropic's tool runner) runs the
same loop for you inside the client library: your functions still run on your machine. A **server-side
tool** (web search, code execution, tool search, an MCP connector) runs on the vendor's machines: you
send one request and the vendor runs the tool and its part of the loop.

Each step hands away work and also visibility and control. The answer may be the same. What you can
still see and stop is not.

Analogy: cooking yourself, a meal kit (you still cook, the recipe is fixed), or a restaurant (you only
see the plate).

## Picture
```mermaid
flowchart LR
    Q["question"] --> H["hand loop<br/>(your code)"]
    Q --> A["SDK helper<br/>(client library)"]
    Q --> S["server tool<br/>(vendor)"]
    H --> T1["your tools<br/>you see every step"]
    A --> T2["your tools<br/>steps hidden in the SDK"]
    S --> T3["vendor's tools<br/>you see the result"]
```

## Step by step
### Setup from earlier lessons
The recorder, the client and Scout's tools from lesson 35, shortened: the six notes, `search_notes`,
the safe `calculator` and `run_tool`, which never raises. Every Gemini call is recorded once to
`../data/llm_cassettes/44_sdk-helpers-and-server-tools.yaml` and replayed after that.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("44_sdk-helpers-and-server-tools")

import ast, json, operator, os, re
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
print("model:", MODEL)

LLM mode: replay (44_sdk-helpers-and-server-tools.yaml)


model: gemini-3.5-flash-lite


The notes and `search_notes`, as in lesson 35 but without paging (six notes fit on one page).

In [2]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]

def search_notes(query: str) -> list[dict]:
    """Search the user's notes for a word or phrase (case-insensitive). Returns the matching notes
    as {id, text}. Use it before answering any question about the notes."""
    q = query.strip().lower()
    if not q:
        raise ValueError("query must not be empty, e.g. query='Mars'")
    return [n for n in NOTES if q in n["text"].lower()]

print(search_notes("day on Mars"))

[{'id': 3, 'text': 'A day on Mars lasts about 24 hours and 39 minutes.'}]


The calculator walks a syntax tree with an allow-list of operators (no `eval`), as in lesson 35.

In [3]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow}

def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError(f"'{ast.unparse(node)[:40]}' is not plain arithmetic: use numbers and + - * / **")

def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression: numbers, + - * / ** and brackets. Returns the number.
    Use it for every calculation instead of doing arithmetic yourself."""
    try:
        return evaluate(ast.parse(expression, mode="eval").body)
    except SyntaxError:
        raise ValueError(f"'{expression}' is not valid arithmetic, e.g. '24*60+39'") from None

TOOLS = {"search_notes": search_notes, "calculator": calculator}
print(calculator("24*60+39"))

1479


`run_tool` from lesson 35: an unknown name or any exception becomes an error result.

In [4]:
def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available tools: {', '.join(tools)}.",
                "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

print(run_tool({"name": "calculator", "args": {"expression": "2^10"}}, TOOLS))

{'content': "ValueError: '2 ^ 10' is not plain arithmetic: use numbers and + - * / **", 'is_error': True}


### 1. The smallest helper: a loop that hides itself
First with a scripted fake, no key. `auto_run` is a toy version of what an SDK helper does: it loops
until the model stops calling tools and returns **only the final text**. It copies two choices the
Gemini SDK makes (we read them in its source, `google/genai/_extra_utils.py`): an exception becomes
`{"error": str(e)}`, losing the exception's type, and the default limit is 10 model calls.

In [5]:
def auto_run(question: str, model, tools: dict, max_calls: int = 10) -> str | None:
    messages = [{"role": "user", "text": question}]
    for _ in range(max_calls):
        reply = model(messages)
        messages.append(reply)
        if not reply.get("tool_calls"):
            return reply["text"]
        for call in reply["tool_calls"]:
            try:
                out = {"result": tools[call["name"]](**call["args"])}
            except Exception as e:
                out = {"error": str(e)}
            messages.append({"role": "tool", "name": call["name"], "content": out})
    return None                           # out of calls: the caller only sees "no answer"

A scripted model for the Mars question: search, a bad expression, the fixed expression, then the answer.

In [6]:
def scripted(replies: list):
    state = {"n": 0}
    def model(messages):
        state["n"] += 1
        reply = replies[state["n"] - 1]
        return reply(messages) if callable(reply) else reply
    return model

def calc(expr): return {"tool_calls": [{"name": "calculator", "args": {"expression": expr}}]}
SCRIPT = [{"tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]},
          calc("24*60+39 minutes"), calc("24*60+39"),
          lambda m: {"text": f"A day on Mars lasts about {m[-1]['content']['result']} minutes."}]
print(auto_run("How many minutes long is a day on Mars?", scripted(SCRIPT), TOOLS))

A day on Mars lasts about 1479 minutes.


The answer is right, and that is all you get. There were four model calls and one failed tool call,
but `auto_run` does not report either. You cannot count steps, log the bad expression, or stop
before a risky tool. Your lesson 35 loop returned `steps`, `errors` and `messages`. Now the same
comparison with the real model.

### 2. The hand loop on Gemini
Lesson 35's loop, shortened to one function. It records each call's input tokens and tool calls in a
`trace`. The tool declarations come from `types.FunctionDeclaration.from_callable`, the same converter
the SDK helper uses on Python functions, so both runs send the same tool definitions.

In [7]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Use the calculator for arithmetic. Answer in one sentence.")
DECLS = [types.Tool(function_declarations=[
    types.FunctionDeclaration.from_callable(client=client, callable=fn) for fn in TOOLS.values()])]
QUESTION = "How many minutes long is a day on Mars?"

def to_part(fc, result: dict) -> types.Part:
    body = {"error": result["content"]} if result["is_error"] else {"output": json.loads(result["content"])}
    return types.Part(function_response=types.FunctionResponse(id=fc.id, name=fc.name, response=body))

print([d.name for d in DECLS[0].function_declarations])

['search_notes', 'calculator']


The loop. `automatic_function_calling` is disabled, so the SDK only makes one request per call.

In [8]:
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def hand_loop(question: str, max_steps: int = 8):
    cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=400,
                                      tools=DECLS, automatic_function_calling=NO_AFC)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]
    trace = []
    for step in range(1, max_steps + 1):
        resp = client.models.generate_content(model=MODEL, contents=contents, config=cfg)
        calls = resp.function_calls or []
        trace.append({"step": step, "input_tokens": resp.usage_metadata.prompt_token_count,
                      "calls": [f"{fc.name}({dict(fc.args)})" for fc in calls]})
        contents.append(resp.candidates[0].content)          # unchanged: keeps the thought signature
        if not calls:
            return resp.text, trace
        results = [run_tool({"name": fc.name, "args": dict(fc.args)}, TOOLS) for fc in calls]
        contents.append(types.Content(role="user", parts=[to_part(fc, r) for fc, r in zip(calls, results)]))
    return None, trace

Run it. Every step is in the trace.

In [9]:
hand_answer, hand_trace = hand_loop(QUESTION)
print("answer:", hand_answer)
for t in hand_trace:
    print(f"step {t['step']}: {t['input_tokens']} input tokens, calls {t['calls'] or 'none (answer)'}")
print("total input tokens:", sum(t["input_tokens"] for t in hand_trace))

answer: A day on Mars is 1,479 minutes long.
step 1: 148 input tokens, calls ["search_notes({'query': 'Mars'})"]
step 2: 216 input tokens, calls ["calculator({'expression': '24 * 60 + 39'})"]
step 3: 242 input tokens, calls none (answer)
total input tokens: 606


Fallback, so later cells run if the cell above was skipped: an empty trace.

In [10]:
if "hand_trace" not in globals():
    hand_answer, hand_trace = None, []
print("hand loop model calls:", len(hand_trace))

hand loop model calls: 3


Gemini searched for "Mars", calculated `24 * 60 + 39` and answered: three model calls, with input
growing from 148 to 242 tokens as the history grew (lesson 08).

### 3. The same question with automatic function calling
Pass the Python **functions** as tools and leave automatic function calling (AFC) on. The SDK builds
the declarations, runs each call, sends the results and loops, up to `maximum_remote_calls`. The
google-genai README says AFC in `models.generate_content` will be removed in the next major version
and recommends **chats** (RESEARCH.md §4), so we use `client.chats`.

In [11]:
afc_cfg = types.GenerateContentConfig(
    system_instruction=SYSTEM, max_output_tokens=400, tools=[search_notes, calculator],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=8))
chat = client.chats.create(model=MODEL, config=afc_cfg)
afc_resp = chat.send_message(QUESTION)
print("answer:", afc_resp.text)
print("input tokens reported:", afc_resp.usage_metadata.prompt_token_count)

answer: A day on Mars is 1,479 minutes long.
input tokens reported: 245


Same answer, one line of code. What is still visible? The chat keeps a history, including the turns the
SDK added. A small helper prints each part.

In [12]:
def describe(part: types.Part) -> str:
    if part.function_call:
        return f"call {part.function_call.name}({dict(part.function_call.args)})"
    if part.function_response:
        return f"result {part.function_response.name} -> {str(part.function_response.response)[:60]}"
    return f"text {part.text!r}"[:80]

afc_history = chat.get_history()
for c in afc_history:
    print(f"{c.role:5}", " | ".join(describe(p) for p in c.parts))

user  text 'How many minutes long is a day on Mars?'
model call search_notes({'query': 'Mars'})
user  result search_notes -> {'result': [{'id': 1, 'text': 'Mars has two small moons, Pho
model call calculator({'expression': '24 * 60 + 39'})
user  result calculator -> {'result': 1479}
model text 'A day on Mars is 1,479 minutes long.'


Fallback for the comparison below.

In [13]:
if "afc_history" not in globals():
    afc_resp, afc_history = None, []
afc_model_calls = sum(1 for c in afc_history if c.role == "model")
print("model turns in the AFC history:", afc_model_calls)

model turns in the AFC history: 3


### 4. What you can no longer see
The history shows the calls and results after the fact, so you can count the turns. Three things are
gone. `usage_metadata` covers
only the **last** model call, not the run. A tool error arrives as `{"error": "<message>"}`, with no
exception type and no `is_error` flag for you to count. And nothing of yours runs **between** steps:
no approval, no budget check, no per-step log.

In [14]:
hand_total = sum(t["input_tokens"] for t in hand_trace)
afc_reported = afc_resp.usage_metadata.prompt_token_count if afc_resp else None
print(f"{'':28}{'hand loop':>12}{'AFC chat':>12}")
print(f"{'model calls you can count':28}{len(hand_trace):>12}{afc_model_calls:>12}")
print(f"{'input tokens you can see':28}{hand_total:>12}{str(afc_reported):>12}")
print(f"{'code between steps':28}{'yes':>12}{'no':>12}")

                               hand loop    AFC chat
model calls you can count              3           3
input tokens you can see             606         245
code between steps                   yes          no


The hand loop's total (606) covers every call. The AFC figure (245) is the last call's input only,
so a cost report built from it would be far too low. You would have to add up the turns yourself.

**The step limit.** With `maximum_remote_calls=1` the SDK makes one request, gets a function call, has
no request left to send the result, and returns that function call. There is no error and no text,
only a printed warning about "non-text parts".

In [15]:
one_call = types.GenerateContentConfig(
    system_instruction=SYSTEM, max_output_tokens=400, tools=[search_notes, calculator],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=1))
limited = client.chats.create(model=MODEL, config=one_call).send_message(QUESTION)
print("text:", limited.text)
print("left over:", [describe(p) for p in limited.candidates[0].content.parts if p.function_call])

text: None
left over: ["call search_notes({'query': 'Mars'})"]


Fallback: the same shape (no text) if the cell above was skipped.

In [16]:
if "limited" not in globals():
    limited = types.GenerateContentResponse()
print("limited.text is None:", limited.text is None)

limited.text is None: True


### 5. A vendor-run tool: Gemini code execution
A **server-side tool** runs on the vendor's machines. Here Gemini writes Python and Google runs it in a
sandbox; no code runs on your laptop. Note what this tool cannot do: it cannot read Scout's notes,
because they live on your machine, so we give it the facts in the prompt.

In [17]:
code_cfg = types.GenerateContentConfig(max_output_tokens=800, automatic_function_calling=NO_AFC,
                                       tools=[types.Tool(code_execution=types.ToolCodeExecution())])
code_resp = client.models.generate_content(
    model=MODEL, config=code_cfg,
    contents="A day on Mars lasts 24 hours and 39 minutes. Write and run Python to compute how many "
             "minutes that is, and how many whole Mars days fit in 1,000,000 minutes.")
for part in code_resp.candidates[0].content.parts:
    if part.executable_code:
        print("CODE (ran on Google's side):\n" + part.executable_code.code)
    elif part.code_execution_result:
        print("RESULT:", part.code_execution_result.outcome, part.code_execution_result.output)
    elif part.text:
        print("TEXT:", part.text[:120], "...")

CODE (ran on Google's side):
# Calculate total minutes in a Mars day
hours_in_day = 24
minutes_extra = 39
total_minutes_per_mars_day = (hours_in_day * 60) + minutes_extra

# Calculate how many whole Mars days fit in 1,000,000 minutes
total_minutes = 1_000_000
whole_mars_days = total_minutes // total_minutes_per_mars_day
remaining_minutes = total_minutes % total_minutes_per_mars_day

print(f"Minutes per Mars day: {total_minutes_per_mars_day}")
print(f"Whole Mars days in 1,000,000 minutes: {whole_mars_days}")
print(f"Remaining minutes: {remaining_minutes}")

RESULT: Outcome.OUTCOME_OK Minutes per Mars day: 1479
Whole Mars days in 1,000,000 minutes: 676
Remaining minutes: 196

TEXT: Here is the Python code and the results of the computation:

```python
# A day on Mars lasts 24 hours and 39 minutes
hou ...


You see the code and its output, but you did not run it, choose the Python version or limit what it
could import. Each code-and-result pair is a loop step the vendor ran inside **one** request.

### 6. Anthropic: a tool runner and dated server-tool types
No Anthropic key was available when this lesson was built, so this step is read-along: we build real
SDK objects offline and **send nothing**. `@beta_tool` turns a function into a tool the beta tool runner
(`client.beta.messages.tool_runner(..., max_iterations=...)`) can call for you. The schema comes from
the type hints and the docstring's `Args:` section.

In [18]:
from anthropic import beta_tool

@beta_tool
def calculator_tool(expression: str) -> str:
    """Evaluate an arithmetic expression.

    Args:
        expression: arithmetic only, e.g. '24*60+39'
    """
    return json.dumps(calculator(expression))

print(json.dumps(calculator_tool.to_dict(), indent=1))

{
 "name": "calculator_tool",
 "description": "Evaluate an arithmetic expression.",
 "input_schema": {
  "additionalProperties": false,
  "properties": {
   "expression": {
    "description": "arithmetic only, e.g. '24*60+39'",
    "title": "Expression",
    "type": "string"
   }
  },
  "required": [
   "expression"
  ],
  "type": "object"
 }
}


Server tools are listed next to your own tools, but with a **dated `type`** instead of a schema.
The date pins the behaviour: a new version gets a new date, and old ones keep working (Anthropic tool
reference, RESEARCH.md §49). The MCP connector is the exception: `mcp_toolset`, versioned by a beta header.

In [19]:
request = {  # built here, not sent
    "model": "claude-sonnet-5-5", "max_tokens": 1024,
    "tools": [calculator_tool.to_dict(),
              {"type": "web_search_20260209", "name": "web_search", "max_uses": 3},
              {"type": "code_execution_20260521", "name": "code_execution"},
              {"type": "tool_search_tool_bm25_20251119", "name": "tool_search_tool_bm25"},
              {"type": "mcp_toolset", "mcp_server_name": "notes"}],
    "mcp_servers": [{"type": "url", "url": "https://notes.example.com/mcp", "name": "notes"}],
    "betas": ["mcp-client-2025-11-20"],
}
for t in request["tools"]:
    date = re.search(r"_(\d{8})$", t.get("type", ""))
    print(f"{t.get('name') or t['mcp_server_name']:22} {t.get('type', 'your function'):32} "
          f"pinned to {date.group(1) if date else '-'}")

calculator_tool        your function                    pinned to -
web_search             web_search_20260209              pinned to 20260209
code_execution         code_execution_20260521          pinned to 20260521
tool_search_tool_bm25  tool_search_tool_bm25_20251119   pinned to 20251119
notes                  mcp_toolset                      pinned to -


Who runs what in that request: Anthropic runs web search, code execution, tool search and the MCP
calls inside one API call; the SDK's tool runner would run `calculator_tool` on your machine. One step
further, **managed agents** (Claude Managed Agents) host the whole loop, its tools and the session on the
vendor's side. You send tasks and read results.

### 7. Choosing who runs the loop
A small rule that captures the trade-offs above. It checks the needs that rule out a helper first.

In [20]:
def who_runs_the_loop(needs: set[str]) -> str:
    if needs & {"approval", "per_step_budget", "per_step_logs", "custom_errors"}:
        return "hand loop: you need code between steps"
    if "own_data_or_side_effects" in needs and "vendor_has_tool" in needs:
        return "hand loop or SDK helper with your tools, plus vendor tools for public data"
    if "vendor_has_tool" in needs:
        return "server-side tool: the vendor runs it (pin the dated type)"
    return "SDK helper: same loop, less code; switch to a hand loop when you need control"

for case in [{"per_step_logs"}, {"vendor_has_tool"}, {"prototype"},
             {"own_data_or_side_effects", "vendor_has_tool"}]:
    print(sorted(case), "->", who_runs_the_loop(case))

['per_step_logs'] -> hand loop: you need code between steps
['vendor_has_tool'] -> server-side tool: the vendor runs it (pin the dated type)
['prototype'] -> SDK helper: same loop, less code; switch to a hand loop when you need control
['own_data_or_side_effects', 'vendor_has_tool'] -> hand loop or SDK helper with your tools, plus vendor tools for public data


## At work
- **Teams prototype with a helper, then take the loop back.** The usual reason is code that must run
  between steps: logging each tool call (→ lesson 49), asking a human before a side effect (→ lesson 52),
  stopping on a cost cap (→ lesson 54).
- **Server tools have their own bill.** Anthropic charges web search per search on top of tokens (tool use
  overview, RESEARCH.md §49), so `max_uses` and budgets matter. → lesson 54
- **Helpers and tool versions change under you.** Gemini AFC in `generate_content` is marked for removal
  in the next SDK major, and server tools are pinned by date. Treat both like a model version. → lesson 53
- **Web results are untrusted input.** A page found by a server-side web search enters the model's
  context like any tool result, and a page can contain instructions. → lesson 51

## What just happened
- The fake `auto_run` answered correctly, but returned only the text: four model calls and one tool
  error were invisible.
- The hand loop on Gemini printed every step's calls and input tokens.
- Gemini's AFC, through `client.chats`, gave the same answer in one call; the chat history still listed
  the calls, but `usage_metadata` covered only the last model call.
- With `maximum_remote_calls=1`, AFC returned an unanswered function call and `text` was `None`, with no
  error.
- Gemini code execution wrote and ran Python on Google's side, inside one request.
- An Anthropic request listed your function next to dated server tools and an MCP toolset, built offline.

## Common mistakes
**1. Trusting the helper's answer after it hit the limit.** The helper stops without an error, so code
that assumes text crashes on `None`, or quietly shows the user nothing.

In [21]:
print("Scout says:", limited.text.strip())

AttributeError: 'NoneType' object has no attribute 'strip'

Fix: check for leftover function calls and treat them as "ran out of steps", as lesson 30's
`step_limit` stop reason did.

In [22]:
parts = limited.candidates[0].content.parts if limited.candidates else []
leftover = [p for p in parts if p.function_call]
stop = "step_limit" if leftover or limited.text is None else "end_of_turn"
print("stop:", stop, "| answer:", limited.text or "(none: ask a human or raise the limit)")

stop: step_limit | answer: (none: ask a human or raise the limit)


**2. Giving the sync helper an `async` tool.** Many I/O tools are `async`. The sync AFC refuses them
before sending anything.

In [23]:
async def search_notes_async(query: str) -> list[dict]:
    """Search the user's notes for a word or phrase."""
    return search_notes(query)

bad = types.GenerateContentConfig(tools=[search_notes_async, calculator])
client.chats.create(model=MODEL, config=bad).send_message(QUESTION)

UnsupportedFunctionError: Function search_notes_async is a coroutine function, which is not supported for automatic function calling. Please manually invoke search_notes_async to get the function response.

Fix: use the async client (`client.aio.chats`), or give the sync helper sync functions. In your own loop
you decide, e.g. with `asyncio.run` per call. The cell below only builds the async chat; it sends nothing.

In [24]:
async_chat = client.aio.chats.create(model=MODEL, config=bad)
print(type(async_chat).__name__, "accepts", [f.__name__ for f in bad.tools])

AsyncChat accepts ['search_notes_async', 'calculator']


## Try it
Get the trace back without giving up the helper. Write `traced(fn, log)`, a wrapper that keeps the
function's name, signature and docstring (`functools.wraps`), and appends `{"tool", "args", "ok"}` to
`log` on every call, re-raising any exception. Wrap both tools, run `auto_run` with `scripted(SCRIPT)`,
and check that `log` has three entries and exactly one has `ok` False.

In [25]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [26]:
llm_replay.stop()

## Key terms
- **SDK helper** — a function in the vendor's client library that runs the tool loop for you on your machine; same loop, less code, less visibility.
- **automatic function calling (AFC)** — google-genai's helper: pass Python functions as tools and the SDK runs each call and loops, up to `maximum_remote_calls` (default 10).
- **tool runner** — Anthropic's beta SDK helper (`client.beta.messages.tool_runner` with `@beta_tool` functions) that runs the tool loop, up to `max_iterations`.
- **server-side tool** — a tool the vendor runs on its own machines inside your API request (web search, code execution, tool search, MCP connector); you see the result, not the run.
- **dated tool type** — a server or Anthropic-defined tool's `type` string ending in a date, such as `web_search_20260209`, that pins its behaviour; new behaviour gets a new date.
- **managed agent** — a vendor-hosted agent: the vendor runs the loop, tools and session; you send tasks and read results.